# Step 4-C: 셀프 플레이 학습 루프 및 아레나 평가 가이드

## 1. DQNAgent 구조 (TODO 1 ~ 9)

### A. `act` 메서드 (TODO 1 ~ 4)
- **Epsilon-Greedy**: `random.random() < epsilon`이면 `random.choice(legal_moves)` 반환
- **상태 텐서 변환**: `env.encode_state()` -> `torch.tensor(..., dtype=torch.float32).unsqueeze(0).to(self.device)`
- **마스크 생성**: `torch.zeros((1, 140), dtype=torch.bool).to(self.device)`, `mask[0, legal_moves] = True`
- **행동 추론**: `with torch.no_grad():` 블록에서 `self.policy_net(state_tensor, mask=mask).argmax(dim=1).item()`

### B. `update` 메서드 (TODO 5 ~ 8)
- **샘플링 조건**: `len(self.buffer) < batch_size`이면 `return 0.0`
- **미니배치 추출**: `batch = self.buffer.sample(batch_size, self.device)`
- **손실 계산**: `loss = compute_double_dqn_loss(self.policy_net, self.target_net, batch, self.gamma)`
- **역전파 4단계**:
  1. `self.optimizer.zero_grad()`
  2. `loss.backward()`
  3. `torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), max_norm=1.0)`
  4. `self.optimizer.step()`

### C. `sync_target` 메서드 (TODO 9)
- `self.target_net.load_state_dict(self.policy_net.state_dict())`

---

## 2. 셀프 플레이 학습 루프 `train_dqn` (TODO 10 ~ 15)

- **TODO 10**: `action = agent.act(env, legal_moves, epsilon)`
- **TODO 11**: `_, reward, done = env.step(action)`
- **TODO 12 (다음 상태 마스크)**:
  - `next_state = env.encode_state()`
  - `next_mask = np.zeros(140, dtype=bool)`
  - 게임이 끝나지 않았다면: `next_mask[env._get_legal_moves()] = True`
- **TODO 13**: `agent.buffer.push(curr_state, action, reward, next_state, done, next_mask)`
- **TODO 14**: `loss_val = agent.update(batch_size)`
- **TODO 15**: `if total_steps % target_sync_freq == 0: agent.sync_target()`


---

## 3. 핵심 테크닉: 액션 관점 정규화 (flip_action)
- **왜 필요한가?**: P1 차례일 때 encode_state()는 보드를 180도 뒤집어 전달하므로, 신경망 입장에서는 **항상 위쪽(액션 0)이 전진**입니다.
- **해결책**:
  1. ct(): P1 턴일 때 
orm_legal_moves = [flip_action(a) for a in legal_moves]로 신경망에 입력하고, 출력된 행동을 lip_action(norm_action)으로 실제 게임판에 되돌려 실행합니다.
  2. 버퍼 저장: gent.buffer.push(...)에는 항상 **신경망 시점에서의 액션(
orm_action)**과 **다음 플레이어 관점의 합법 마스크(
ext_norm_legal)**를 저장합니다.
